In [2]:
import pandas as pd
import re
import sys
from pathlib import Path

In [3]:
s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t')
s2 = pd.read_csv("../student_resource/dataset/train/train_source2.tsv", sep='\t')
s3 = pd.read_csv("../student_resource/dataset/train/train_source3.tsv", sep='\t')
g = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep='\t') 

In [4]:
sys.path.append("..")
from src.normalization import extract_geo_layers

In [5]:
def find_optimal_threshold(s1, s2, s3, thresholds=[10000, 12000, 15000, 20000]):
    print("Combining datasets for global threshold calculation...")
    
    combined = pd.concat([
        s1[['country', 'business_address']], 
        s2[['country', 'business_address']], 
        s3[['country', 'business_address']]
    ]).dropna(subset=['business_address']).copy()
    
    print("Extracting base layers (this takes a moment)...")
    geo_data = combined.apply(lambda row: extract_geo_layers(row['business_address'], row['country']), axis=1)
    geo_df = pd.DataFrame(geo_data.tolist())
    
    def assign_base(row):
        if row['pin']: return f"{row['country']}_{row['pin']}"
        elif row['chunk_1']: return f"{row['country']}_{row['chunk_1']}"
        return f"{row['country']}_unknown"
        
    print("Assigning base keys...")
    sim_df = pd.DataFrame({
        'base_key': geo_df.apply(assign_base, axis=1),
        'chunk_1': geo_df['chunk_1'],
        'chunk_2': geo_df['chunk_2'],
        'country': geo_df['country']
    })
    
    base_counts = sim_df['base_key'].value_counts()
    
    print("\n" + "="*30)
    print("SWEEPING THRESHOLDS")
    print("="*30)
    
    for thresh in thresholds:
        massive_keys = set(base_counts[base_counts > thresh].index)
        
        def simulate_split(row):
            key = row['base_key']
            if key in massive_keys and not re.search(r'\d{5,6}$', key):
                chunk_2 = row['chunk_2']
                if chunk_2:
                    return f"{row['country']}_{chunk_2}_{row['chunk_1']}"
            return key
            
        final_keys = sim_df.apply(simulate_split, axis=1)
        final_counts = final_keys.value_counts()
        
        print(f"\n[ Threshold: {thresh} ]")
        print(f"-> Massive Blocks Split: {len(massive_keys)}")
        print(f"-> Max Block Size: {final_counts.max()} records")
        print(f"-> No of Blocks: {len(final_counts)}")
        print("-> Top 3 Largest Blocks Post-Split:")
        print(final_counts.head(3).to_string())

# Execute the sweep on your raw sample dataframes
find_optimal_threshold(s1, s2, s3)

Combining datasets for global threshold calculation...
Extracting base layers (this takes a moment)...
Assigning base keys...

SWEEPING THRESHOLDS


KeyboardInterrupt: 

In [6]:
import pandas as pd
from src.blocking import run_multi_index_generation, run_multi_source_geo_blocking

def generate_global_candidate_pool(s1, s2, s3, top_k=15):
    """Executes Index 1 & 2 across all pairs and standardizes columns."""
    print("="*40)
    print("STAGE 1-3: GLOBAL BM25 INDICES")
    print("="*40)
    
    pairs_12 = run_multi_index_generation(s1, s2, top_k)
    pairs_13 = run_multi_index_generation(s1, s3, top_k)
    pairs_23 = run_multi_index_generation(s2, s3, top_k)
    
    std_cols = ['entity_A', 'entity_B', 'bm25_text_score', 'bm25_phonetic_score']
    
    if not pairs_12.empty: pairs_12.columns = std_cols
    if not pairs_13.empty: pairs_13.columns = std_cols
    if not pairs_23.empty: pairs_23.columns = std_cols
    
    master_pool = pd.concat([pairs_12, pairs_13, pairs_23], ignore_index=True)
    return master_pool.drop_duplicates(subset=['entity_A', 'entity_B'])

# 1. Run Global Search (Index 1 & 2)
global_candidates_df = generate_global_candidate_pool(s1_normalized, s2_normalized, s3_normalized, top_k=15)

# 2. Run Local Geo Search (Index 3)
geo_candidates_df = run_multi_source_geo_blocking(s1_normalized, s2_normalized, s3_normalized, top_k=15)

# 3. Merge All Nets Together
print("\nMerging Local Geo matches into Master Candidate Pool...")
final_master_pool = pd.merge(
    global_candidates_df, 
    geo_candidates_df, 
    on=['entity_A', 'entity_B'], 
    how='outer'
).fillna(0.0)

print(f"Final Candidate Pool Size: {len(final_master_pool)}")

# 4. Final Validation check 
captured, missed = validate_blocking_recall(
    final_master_pool, 
    g, 
    s1_normalized, 
    s2_normalized, 
    s3_normalized
)

NameError: name 's1_normalized' is not defined